# Payment Pulse: Merchant Payment Disruption Early Warning

**Theme:** Real-Time AI for Critical Decisions — detect and respond to payment-service disruption as transaction events arrive.

This project uses the card transaction CSV to monitor changes in transaction volume and recorded processing errors by city and hour. It forecasts expected activity with XGBoost and identifies unusual multivariate patterns with scikit-learn Isolation Forest. A flagged period means ‘investigate service health’; it is not proof of an outage or fraud. The `Is Fraud?` field is deliberately ignored, so this is a different problem from the mandatory fraud-detection lab. A human operator decides what to do; the notebook never blocks or reroutes payments.

## Visual guide: how Payment Pulse works

![Payment Pulse early-warning workflow](assets/payment-pulse-overview.png)

Read the illustration from left to right: transaction events are summarized into area-level activity, models compare the observed pattern with expected behavior, and a human operator reviews any warning. Orange markers indicate example error signals; the artwork is explanatory, not real measured output.

In [ ]:
# Display the generated visual from the project assets folder.
from IPython.display import Image, display
display(Image(filename='assets/payment-pulse-overview.png', width=1000))

## Payment Pulse: input-to-output flow

This diagram shows how transaction CSV rows become short time-window signals, how the two models analyze them, and how the resulting alert supports a human decision. The output is a review cue, not an automatic action.

In [ ]:
# Import the notebook image display helper and show the generated workflow diagram.
from IPython.display import Image, display
from pathlib import Path

flow_image = Path('assets/payment-pulse-transaction-flow.png')
if not flow_image.is_file():
    raise FileNotFoundError('Keep the assets folder beside this notebook when uploading it to JupyterLab.')
display(Image(filename=str(flow_image), width=1100))

## 1. Load a bounded dataset sample

The full CSV is large, so this notebook reads a configurable sample. On L1CC, place it in JupyterLab's `shared` folder. The code also checks the current working folder as a fallback. We exclude the fraud label and do not use it as a target or feature.

In [ ]:
from pathlib import Path
import json, math, time, warnings
import numpy as np
import pandas as pd

candidate_paths = [Path.home() / 'shared' / 'transactions_history.csv', Path('transactions_history.csv'), Path('card_transaction.v1.csv')]
CSV_PATH = next((p for p in candidate_paths if p.is_file()), None)
if CSV_PATH is None:
    raise FileNotFoundError('Put the transaction CSV in ~/shared/transactions_history.csv or the notebook folder.')

SAMPLE_ROWS = 150_000  # small bounded training/demo sample; increase only after checking VM memory
raw = pd.read_csv(CSV_PATH, nrows=SAMPLE_ROWS, dtype={'Merchant Name': 'string', 'Merchant City': 'string', 'Merchant State': 'string', 'Errors?': 'string'})
required = {'User','Card','Year','Month','Day','Time','Amount','Merchant City','Merchant State','Errors?'}
missing = required - set(raw.columns)
if missing: raise ValueError(f'Missing expected columns: {sorted(missing)}')
raw = raw.drop(columns=['Is Fraud?'], errors='ignore')
print(f'Dataset: {CSV_PATH}')
print(f'Loaded sample: {len(raw):,} rows × {len(raw.columns)} columns')
print('Fraud label present in analysis data:', 'Is Fraud?' in raw.columns)
display(raw.head(5))

## 2. Dataset statistics and data quality

Check missing values, time coverage, locations, and recorded error frequency. These are descriptive dataset statistics, not model performance results.

In [ ]:
print('Rows / columns:', raw.shape)
print('Unique cities:', raw['Merchant City'].nunique(dropna=True))
print('Unique states:', raw['Merchant State'].nunique(dropna=True))
print('Recorded error rows:', raw['Errors?'].fillna('').str.strip().ne('').sum())
print('Missing values (top 12):')
display(raw.isna().sum().sort_values(ascending=False).head(12).to_frame('missing_rows'))
display(raw[['Merchant City','Merchant State','Errors?']].describe(include='all'))

## 3. Prepare hourly area-level operating signals

Each observation is one **city/state/hour**. Features summarize event volume, number of affected cards, error rate, and typical transaction amount. Grouping by location and time gives a service-health view instead of a per-person fraud score. The sample is bounded, so rare or small cities may have too few observations for reliable scoring.

In [ ]:
def make_hourly_windows(frame):
    d = frame.copy()
    time_text = d['Time'].astype('string').str.slice(0, 5)
    date_text = (d['Year'].astype('Int64').astype('string') + '-' + d['Month'].astype('Int64').astype('string').str.zfill(2) + '-' + d['Day'].astype('Int64').astype('string').str.zfill(2))
    d['event_time'] = pd.to_datetime(date_text + ' ' + time_text, errors='coerce')
    d = d.dropna(subset=['event_time'])
    d['window_start'] = d['event_time'].dt.floor('h')
    d['city'] = d['Merchant City'].fillna('Unknown').astype(str).str.strip()
    d['state'] = d['Merchant State'].fillna('Unknown').astype(str).str.strip()
    d['area'] = d['city'] + ' | ' + d['state']
    d['has_error'] = d['Errors?'].fillna('').astype(str).str.strip().ne('').astype(int)
    d['amount_value'] = pd.to_numeric(d['Amount'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False), errors='coerce')
    windows = (d.groupby(['area','city','state','window_start'], as_index=False, observed=True)
        .agg(event_count=('Card','size'), error_count=('has_error','sum'), unique_cards=('Card','nunique'), mean_amount=('amount_value','mean'), amount_std=('amount_value','std')))
    windows['error_rate'] = windows['error_count'] / windows['event_count'].clip(lower=1)
    windows['mean_amount'] = windows['mean_amount'].fillna(0)
    windows['amount_std'] = windows['amount_std'].fillna(0)
    windows['hour'] = windows['window_start'].dt.hour
    windows['day_of_week'] = windows['window_start'].dt.dayofweek
    return windows.sort_values(['area','window_start']).reset_index(drop=True)

windows = make_hourly_windows(raw)
print(f'Created {len(windows):,} hourly city windows across {windows.area.nunique():,} areas.')
display(windows.head(8))

## 4. Explore traffic and error patterns (graphs)

These plots show the data the models will analyze. A spike in recorded errors or a sudden activity drop can signal a service issue, but this historical sample has no verified outage labels.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(windows['event_count'], bins=40, color='#2878b5', edgecolor='white')
axes[0].set(title='Hourly transaction volume by city', xlabel='Transactions in one city-hour', ylabel='Number of city-hour windows')
axes[1].hist(windows['error_rate'], bins=30, color='#e07a37', edgecolor='white')
axes[1].set(title='Recorded error-rate distribution', xlabel='Errors / transactions', ylabel='Number of city-hour windows')
plt.tight_layout(); plt.show()

top_areas = windows.groupby('area')['event_count'].sum().sort_values(ascending=False).head(10).sort_values()
plt.figure(figsize=(9,5)); top_areas.plot.barh(color='#4c9f70')
plt.title('Top 10 areas by transaction volume in the loaded sample'); plt.xlabel('Transactions'); plt.ylabel('City | State'); plt.tight_layout(); plt.show()

## 5. Project flow diagram

This image summarizes the decision-support workflow: events become hourly aggregates, two models produce independent signals, and a human operator reviews alerts.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 3.2)); ax.set_xlim(0,12); ax.set_ylim(0,3); ax.axis('off')
boxes = [(0.3,'CSV transaction / events'),(2.6,'Hourly city features'),(5.0,'XGBoost volume forecast'),(7.4,'IsolationForest anomaly score'),(9.8,'Human review / service health')]
for x,label in boxes:
    box=FancyBboxPatch((x,1.05),1.8,0.9,boxstyle='round,pad=0.15',fc='#e8f2fa',ec='#2878b5',lw=1.5); ax.add_patch(box); ax.text(x+0.9,1.5,label,ha='center',va='center',fontsize=10)
for x in [2.1,4.5,6.9,9.3]: ax.add_patch(FancyArrowPatch((x,1.5),(x+0.45,1.5),arrowstyle='->',mutation_scale=14,color='#334'))
ax.text(6,0.35,'No fraud label • no automatic block/reroute • alert is an operational signal, not a confirmed outage',ha='center',fontsize=10,color='#663c00')
plt.tight_layout(); plt.show()

## 6. Train two models on a small sample

**Model A — XGBoost:** forecast current hourly transaction volume from the prior hour and a short rolling baseline. Large negative forecast errors can be an early traffic-drop signal.

**Model B — scikit-learn Isolation Forest:** learn the shape of typical hourly volume, error rate, card count, and amounts; unusually different windows receive a lower normality score.

We split by time so later windows form the holdout. These are unsupervised/forecasting models because the CSV does not contain verified service-outage labels.

In [ ]:
from xgboost import XGBRegressor
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

def add_lags(frame):
    f=frame.copy().sort_values(['area','window_start'])
    g=f.groupby('area', sort=False)
    f['lag_volume']=g['event_count'].shift(1)
    f['lag_error_rate']=g['error_rate'].shift(1)
    f['lag_cards']=g['unique_cards'].shift(1)
    f['lag_volume_mean4']=g['event_count'].transform(lambda s:s.shift(1).rolling(4,min_periods=1).mean())
    f['area_code']=pd.factorize(f['area'])[0].astype(float)
    return f.dropna(subset=['lag_volume','lag_error_rate','lag_cards']).reset_index(drop=True)

model_data=add_lags(windows)
FORECAST_COLS=['area_code','hour','day_of_week','lag_volume','lag_error_rate','lag_cards','lag_volume_mean4']
ANOMALY_COLS=['event_count','error_rate','unique_cards','mean_amount','amount_std','hour','day_of_week']
if len(model_data)<50: raise ValueError('Too few city-hour windows to train. Increase SAMPLE_ROWS and rerun from section 1.')
cutoff=model_data['window_start'].quantile(0.8)
train=model_data[model_data['window_start']<cutoff].copy()
test=model_data[model_data['window_start']>=cutoff].copy()
if len(train)<20 or len(test)<10: raise ValueError('Temporal split too small. Increase SAMPLE_ROWS.')

xgb_model=XGBRegressor(n_estimators=120,max_depth=4,learning_rate=0.07,subsample=0.85,colsample_bytree=0.85,objective='reg:squarederror',tree_method='hist',n_jobs=2,random_state=42)
xgb_model.fit(train[FORECAST_COLS],train['event_count'])
train['forecast']=np.maximum(0,xgb_model.predict(train[FORECAST_COLS]))
test['forecast']=np.maximum(0,xgb_model.predict(test[FORECAST_COLS]))
volume_mae=mean_absolute_error(test['event_count'],test['forecast'])

scaler=StandardScaler()
X_if_train=scaler.fit_transform(train[ANOMALY_COLS].astype(float))
isolation=IsolationForest(n_estimators=120,contamination='auto',random_state=42,n_jobs=2)
isolation.fit(X_if_train)
if_train_score=isolation.score_samples(X_if_train)
if_threshold=float(np.quantile(if_train_score,0.025))
train_residual=train['event_count'].to_numpy()-train['forecast'].to_numpy()
drop_threshold=float(min(-2.0,np.quantile(train_residual,0.05)))

print(f'Training city-hour windows: {len(train):,}; later holdout windows: {len(test):,}')
print(f'XGBoost holdout volume MAE: {volume_mae:.3f} transactions/window')
print(f'Isolation Forest low-score threshold (2.5th train percentile): {if_threshold:.4f}')
print(f'Forecast drop threshold (train residual 5th percentile, capped below -2): {drop_threshold:.3f}')
print('The two anomaly thresholds are calibrated from this sample; they are demo settings, not operational policy.')

## 7. Analyze holdout forecasts and alerts

Each city-hour gets two independent checks: a negative volume forecast residual and an Isolation Forest outlier score. An alert means an operator should inspect gateway/terminal health and corroborating data. It is not a confirmed incident.

In [ ]:
def score_windows(frame):
    out=frame.copy()
    out['forecast']=np.maximum(0,xgb_model.predict(out[FORECAST_COLS]))
    out['volume_residual']=out['event_count']-out['forecast']
    out['isolation_score']=isolation.score_samples(scaler.transform(out[ANOMALY_COLS].astype(float)))
    out['traffic_drop']=(out['forecast']>=3)&(out['volume_residual']<=drop_threshold)&(out['event_count']<=0.4*out['forecast'])
    out['error_spike']=(out['event_count']>=3)&(out['error_rate']>=0.30)&(out['error_rate']>=out['lag_error_rate']+0.20)
    out['multivariate_anomaly']=out['isolation_score']<if_threshold
    out['alert']=out['traffic_drop']|out['error_spike']|out['multivariate_anomaly']
    out['status']=np.select([out['traffic_drop']|out['error_spike'],out['multivariate_anomaly']],['EARLY WARNING','WATCH'],default='NORMAL')
    out['action']='Continue routine monitoring'
    out.loc[out['status']=='WATCH','action']='Review area-level activity and compare with service telemetry'
    out.loc[out['status']=='EARLY WARNING','action']='Check processor, terminal, and network health; operator decides escalation'
    return out

scored_test=score_windows(test)
print('Holdout windows:',len(scored_test),' | model alerts:',int(scored_test['alert'].sum()))
display(scored_test.sort_values(['status','volume_residual']).tail(12)[['window_start','city','state','event_count','forecast','error_rate','isolation_score','status','action']])

## 8. Precision, recall, and F1 with a transparent synthetic incident check

The CSV has no verified outage labels, so real-world outage precision/recall/F1 cannot be calculated from it. To check whether the rules react to known patterns, we create a **synthetic benchmark**: normal held-out windows are negatives; copies with an 85% traffic drop and an 80% recorded-error rate are labeled as injected disruptions. Metrics below measure detection of these injected examples only. They are not evidence of real outage accuracy.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

n=min(500,len(test))
normal_eval=test.sample(n=n,random_state=7).copy()
incident_eval=normal_eval.copy()
incident_eval['event_count']=np.floor(incident_eval['event_count']*0.15).astype(int)
incident_eval['error_count']=np.maximum(incident_eval['error_count'],np.ceil(incident_eval['event_count'].clip(lower=1)*0.80)).astype(int)
incident_eval['error_rate']=0.80
eval_frame=pd.concat([normal_eval,incident_eval],ignore_index=True)
y_true=np.r_[np.zeros(n,dtype=int),np.ones(n,dtype=int)]
eval_scored=score_windows(eval_frame)
y_pred=eval_scored['alert'].astype(int).to_numpy()
precision=precision_score(y_true,y_pred,zero_division=0)
recall=recall_score(y_true,y_pred,zero_division=0)
f1=f1_score(y_true,y_pred,zero_division=0)
print('SYNTHETIC INJECTION BENCHMARK — not real outage performance')
print(f'Precision: {precision:.3f}  |  Recall: {recall:.3f}  |  F1: {f1:.3f}')
print(classification_report(y_true,y_pred,target_names=['Normal held-out sample','Injected disruption'],zero_division=0))
print('Confusion matrix [normal, injected disruption]:\n',confusion_matrix(y_true,y_pred))

## 9. Graph the model and incident-check results

The forecast chart compares observed volume with expected volume for one area. The final graph shows how the alert rule classified normal and injected windows. Include these figures in your demo and say clearly that the incident cases were simulated.

In [ ]:
area_for_plot=test['area'].value_counts().index[0]
plot_area=scored_test[scored_test['area']==area_for_plot].sort_values('window_start').tail(72)
fig,axes=plt.subplots(2,1,figsize=(12,8))
axes[0].plot(plot_area['window_start'],plot_area['event_count'],label='Observed transactions',marker='o',ms=3)
axes[0].plot(plot_area['window_start'],plot_area['forecast'],label='XGBoost expected volume',marker='.',ms=4)
alerts=plot_area[plot_area['alert']]
axes[0].scatter(alerts['window_start'],alerts['event_count'],color='red',label='Alert window',zorder=4)
axes[0].set(title=f'Observed vs forecast activity — {area_for_plot}',ylabel='Transactions/hour'); axes[0].legend(); axes[0].tick_params(axis='x',rotation=25)
axes[1].plot(plot_area['window_start'],plot_area['error_rate'],color='#d97706',marker='.',label='Recorded error rate')
axes[1].axhline(.30,color='red',ls='--',label='Demo error-spike rule'); axes[1].set(xlabel='Hour',ylabel='Error rate',title='Operational error signals'); axes[1].legend(); axes[1].tick_params(axis='x',rotation=25)
plt.tight_layout(); plt.show()

cm=confusion_matrix(y_true,y_pred,labels=[0,1])
fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,cmap='Blues')
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Predicted normal','Predicted alert'],yticklabels=['Actual normal','Injected disruption'],title='Synthetic incident-check confusion matrix')
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

## 10. Live-style replay and clearly printed output

This replays chronological hourly aggregates from the loaded sample with a short delay. It simulates events arriving in time; the source is historical data, not a live payment feed. The output includes an operational status and suggested human action. No customer names/card numbers are included in alerts.

In [ ]:
REPLAY_LIMIT=100  # number of city-hour windows shown
DELAY_SECONDS=0.03
replay_rows=scored_test.sort_values('window_start').head(REPLAY_LIMIT)
for i,(_,r) in enumerate(replay_rows.iterrows(),start=1):
    print(f"[{i:03d}] {r.window_start} | {r.city}, {r.state} | volume={int(r.event_count)} expected={r.forecast:.1f} errors={r.error_rate:.1%} | {r.status} | {r.action}")
    time.sleep(DELAY_SECONDS)

print('\nRUN SUMMARY')
print(f'Windows replayed: {len(replay_rows)}')
print(f'Early warnings: {(replay_rows.status=="EARLY WARNING").sum()}')
print(f'Watch alerts: {(replay_rows.status=="WATCH").sum()}')
print(f'Holdout forecast MAE: {volume_mae:.3f} transactions/hour')
print(f'Synthetic benchmark precision / recall / F1: {precision:.3f} / {recall:.3f} / {f1:.3f}')
print('Outcome: use signals to check service health; an operator owns escalation.')

## 11. Optional Granite summary (watsonx.ai)

Granite can explain aggregate model results for an operator. This step is optional and requires your own IBM Cloud API key and watsonx.ai project ID. The SSH `.pem` key is unrelated. The prompt sends only aggregate statistics; never send raw transaction records, card/user identifiers, or secrets. The LLM is not the detector and cannot confirm an outage.

In [ ]:
import getpass, requests
api_key=getpass.getpass('IBM Cloud API key (blank to skip): ')
project_id=getpass.getpass('watsonx.ai project ID (blank to skip): ')
region=input('watsonx.ai region [us-south]: ').strip() or 'us-south'

if not api_key or not project_id:
    print('Skipped Granite: credentials were not supplied.')
else:
    summary={'sample_rows':int(len(raw)),'city_hour_windows':int(len(windows)),'holdout_windows':int(len(test)),'holdout_volume_mae':round(float(volume_mae),3),'synthetic_precision':round(float(precision),3),'synthetic_recall':round(float(recall),3),'synthetic_f1':round(float(f1),3),'sample_alerts':int(scored_test.alert.sum())}
    token_r=requests.post('https://iam.cloud.ibm.com/identity/token',data={'grant_type':'urn:ibm:params:oauth:grant-type:apikey','apikey':api_key},headers={'Content-Type':'application/x-www-form-urlencoded'},timeout=30); token_r.raise_for_status()
    chat_r=requests.post(f'https://{region}.ml.cloud.ibm.com/ml/v1/text/chat?version=2025-10-25',headers={'Authorization':'Bearer '+token_r.json()['access_token'],'Content-Type':'application/json','Accept':'application/json'},json={'model_id':'ibm/granite-4-h-small','project_id':project_id,'messages':[{'role':'user','content':'Write 3 concise operator-facing bullets interpreting these aggregate payment-service monitoring results. Do not claim a confirmed outage, do not discuss fraud, state that F1/precision/recall are from injected synthetic scenarios only, and recommend independent checks. Data: '+json.dumps(summary)}],'max_completion_tokens':300,'temperature':0.2},timeout=60); chat_r.raise_for_status()
    print(json.dumps(chat_r.json(),indent=2)[:5000])

## 12. Submission notes: deployment and security

- The project uses the provided LinuxONE JupyterLab s390x environment and its preinstalled XGBoost, pandas, scikit-learn, and matplotlib packages. The GitHub lab package list shows these libraries are included in the environment ([package list](https://github.com/linuxone-community-cloud/jupyter-lab-ml/blob/main/packages.txt)).
- For a containerized service, package the event aggregation and model-scoring functions behind a small API and deploy it on the VM with Docker/Podman. Mount the CSV and model artifacts read-only, run as a non-root user, drop unnecessary capabilities, set memory/CPU limits, and expose the service through an SSH tunnel.
- Keep SSH/API credentials out of notebooks, images, archives, and Git. Only aggregate area-level counts should go to Granite.
- Graphs and reported metrics must be regenerated from your VM run. No real incident labels exist in this CSV, so the synthetic precision/recall/F1 must be labeled as such. A genuine outage evaluation needs verified incident timestamps from an independent source.
- This prototype supports human triage. It does not automatically block, reroute, declare an outage, or replace payment operations monitoring.

## Final results at a glance

At the end of a successful run, the notebook shows: sample size and data-quality statistics; number of hourly city windows; XGBoost forecast MAE; Isolation Forest and forecast-drop thresholds; precision, recall, and F1 on the explicitly synthetic injected-disruption benchmark; a forecast/error time-series graph; a confusion-matrix graph; and printed replay alerts with recommended human actions. Save the notebook with outputs after running it on L1CC for your demo evidence.